In [1]:
import torch
from keras.datasets import mnist
import matplotlib.pyplot as plt
import cv2
import torch.nn as nn 
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchinfo import summary

In [2]:
""""
self.conv1 = nn.Conv2d(
    in_channels=1,    # input channel (e.g. grayscale image)
    out_channels=6,   # কতগুলো ফিচার ম্যাপ বের হবে
    kernel_size=5,    # প্রতিটা ফিল্টারের আকার 5x5
    stride=1,         # এক ধাপ করে ফিল্টার সরে যাবে
    padding=2         # চারপাশে 2 পিক্সেল জিরো প্যাড যোগ হবে
)
"""

'"\nself.conv1 = nn.Conv2d(\n    in_channels=1,    # input channel (e.g. grayscale image)\n    out_channels=6,   # কতগুলো ফিচার ম্যাপ বের হবে\n    kernel_size=5,    # প্রতিটা ফিল্টারের আকার 5x5\n    stride=1,         # এক ধাপ করে ফিল্টার সরে যাবে\n    padding=2         # চারপাশে 2 পিক্সেল জিরো প্যাড যোগ হবে\n)\n'

In [ ]:
# LetNet-5 Model Architecture
class LetNet5(nn.Module):
    def __init__(self):
        super(LetNet5, self).__init__()
        # conv2d(input channel, output_channels, kernel, stried, padding) # output_channels means feature map
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2) # 28x28 -> 28x28
        self.relu = nn.ReLU() # activation function
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2) # Pooled feature map 14x14
        
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1) # conv layer 10x10
        self.fc1 = nn.Linear(16*5*5, 120)  # 5x5x16 = 400
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x))) # Conv1 -> ReLU -> Pool
        x = self.pool(self.relu(self.conv2(x))) # Conv2 -> ReLU -> Pool
        x = x.view(-1, 16*5*5)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [3]:
model = LetNet5()
# input_size=(batch_size, channels, height, width)
summary(model, (1, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
LetNet5                                  [1, 10]                   --
├─Conv2d: 1-1                            [1, 6, 28, 28]            156
├─ReLU: 1-2                              [1, 6, 28, 28]            --
├─AvgPool2d: 1-3                         [1, 6, 14, 14]            --
├─Conv2d: 1-4                            [1, 16, 10, 10]           2,416
├─ReLU: 1-5                              [1, 16, 10, 10]           --
├─AvgPool2d: 1-6                         [1, 16, 5, 5]             --
├─Linear: 1-7                            [1, 120]                  48,120
├─ReLU: 1-8                              [1, 120]                  --
├─Linear: 1-9                            [1, 84]                   10,164
├─ReLU: 1-10                             [1, 84]                   --
├─Linear: 1-11                           [1, 10]                   850
Total params: 61,706
Trainable params: 61,706
Non-trainable params: 0
To

In [61]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3018,)) # MNIST dataset এর mean=0.1307, std=0.3081
])

In [62]:
transform

Compose(
    ToTensor()
    Normalize(mean=(0.1307,), std=(0.3018,))
)

In [63]:
batch_size = 64
epochs = 20
learning_rate = 0.001

In [34]:
train_dataset = torchvision.datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root="./data", train=False, transform=transform, download=True)

100%|██████████| 9.91M/9.91M [00:23<00:00, 415kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 102kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 663kB/s] 
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


In [64]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [65]:
# model, loss and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [46]:
model = LetNet5().to(device)
model

LetNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (relu): ReLU()
  (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [68]:
criterion = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model.parameters(), lr=learning_rate)

In [69]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [ ]:
# Training
train_losses = []
for epoch in range(epochs):
    model.train()
    running_loss = 0.00

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        # pass image batch
        prediction = model(images)
        loss = criterion(prediction, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    print(f"Epoch {epoch+1}/ {epochs}, Loss: {avg_loss:.4f}")

Epoch 1/ 20, Loss: 0.2665
Epoch 2/ 20, Loss: 0.0823
Epoch 3/ 20, Loss: 0.0579
Epoch 4/ 20, Loss: 0.0464
Epoch 5/ 20, Loss: 0.0376
Epoch 6/ 20, Loss: 0.0322
Epoch 7/ 20, Loss: 0.0275
Epoch 8/ 20, Loss: 0.0237
Epoch 9/ 20, Loss: 0.0209
Epoch 10/ 20, Loss: 0.0184
Epoch 11/ 20, Loss: 0.0167
Epoch 12/ 20, Loss: 0.0138
Epoch 13/ 20, Loss: 0.0143
Epoch 14/ 20, Loss: 0.0122
Epoch 15/ 20, Loss: 0.0121
Epoch 16/ 20, Loss: 0.0099
Epoch 17/ 20, Loss: 0.0089
Epoch 18/ 20, Loss: 0.0086
Epoch 19/ 20, Loss: 0.0103
Epoch 20/ 20, Loss: 0.0079


In [50]:
# Evaluation
test_accuracy = []
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total 
    test_accuracy.append(accuracy)

print(f"Test accuracy is : {accuracy:.2f}%")    

Test accuracy is : 99.02%
